# IMPORTS

In [ ]:
import pickle as pkl
from torch import nn
import os
import random
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import torch
import torch.nn.functional as F
import plotly.graph_objects as go
from matplotlib import cm
import wandb

# PARAMS

In [ ]:
# PARAMS
max_length = 50

parent_folder = "nanos_networkx_small"  # Update this to your data path - this is relative!
chunk_length = 50
max_proteins = 3000  # Limit number of proteins for faster execution
batch_size = 32
lr = 1e-4 # learning rate
num_epochs = 300
min_epochs = 30
patience = 10

# More aggressive subsequence parameters
min_subseq_length = 20  # Even smaller minimum subsequence length
step_size = 1  # Much smaller step size for more overlap
subgraph_limit = max_proteins * 300 # max number of subgraphs


model_path = "dual_output_gran_model_residual.pt"  # Path for saving/loading model


# Set device
def get_device():
    """
    Returns the best available device for PyTorch operations.
    Priority: CUDA GPU > MPS (Apple Silicon) > CPU
    """
    if torch.cuda.is_available():
        return torch.device("cuda")
    elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return torch.device("mps")
    else:
        return torch.device("cpu")

device = get_device()
print(f"Using device: {device}")




# MODEL CLASSES - to be split out

In [ ]:
#split out
class GraphAttention(nn.Module):
    def __init__(self, in_features, out_features, n_heads=4, dropout=0.1, alpha=0.2):
        super(GraphAttention, self).__init__()
        self.in_features = in_features
        self.out_features = out_features  # out_features_per_head
        self.n_heads = n_heads
        self.dropout = dropout
        self.alpha = alpha

        # Linear transformations
        self.W = nn.Linear(in_features, n_heads * out_features, bias=False)
        self.a = nn.Linear(2 * out_features, 1, bias=False)

        # Initialize parameters
        nn.init.xavier_uniform_(self.W.weight)
        nn.init.xavier_uniform_(self.a.weight)

        self.leakyrelu = nn.LeakyReLU(self.alpha)
        self.dropout_layer = nn.Dropout(self.dropout)

    def forward(self, h, adj):
        # h: [batch_size, num_nodes, in_features] (32, 50, 128)
        # adj: [batch_size, num_nodes, num_nodes] (32, 50, 50)

        batch_size = h.size(0)
        num_nodes = h.size(1)

        # Linear transformation
        Wh = self.W(h)  # [32, 50, n_heads*out_features] (32,50,4*32=128)
        Wh = Wh.view(batch_size, num_nodes, self.n_heads, self.out_features)  # [32,50,4,32]
        Wh = Wh.permute(0, 2, 1, 3)  # [32,4,50,32]

        # Compute attention coefficients
        Wh_repeated_i = Wh.unsqueeze(3).expand(-1, -1, -1, num_nodes, -1)  # [32,4,50,50,32]
        Wh_repeated_j = Wh.unsqueeze(2).expand(-1, -1, num_nodes, -1, -1)  # [32,4,50,50,32]
        concat = torch.cat([Wh_repeated_i, Wh_repeated_j], dim=-1)  # [32,4,50,50,64]

        e = self.leakyrelu(self.a(concat).squeeze(-1))  # [32,4,50,50]

        # Mask attention coefficients
        zero_vec = -9e15 * torch.ones_like(e)
        adj = adj.unsqueeze(1)  # [32,1,50,50]
        attention = torch.where(adj > 0, e, zero_vec)
        attention = F.softmax(attention, dim=-1)  # [32,4,50,50]
        attention = self.dropout_layer(attention)

        # Apply attention
        h_prime = torch.matmul(attention, Wh)  # [32,4,50,32]
        h_prime = h_prime.permute(0, 2, 1, 3).contiguous()  # [32,50,4,32]
        h_prime = h_prime.view(batch_size, num_nodes, -1)  # [32,50,128]

        return h_prime

#split out
class DualOutputGRAN(nn.Module):
    """
    Graph Recurrent Attention Network that generates both adjacency matrices and amino acid sequences.
    """
    def __init__(self, node_features=38, hidden_dim=128, num_layers=2,
                 n_heads=4, dropout=0.1, amino_acid_vocab_size=22):
        super(DualOutputGRAN, self).__init__()
        self.node_features = node_features
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers
        self.n_heads = n_heads
        self.amino_acid_vocab_size = amino_acid_vocab_size

        assert hidden_dim % n_heads == 0, "hidden_dim must be divisible by n_heads"
        self.out_features_per_head = hidden_dim // n_heads

        # Node feature embedding
        self.node_embedding = nn.Linear(node_features, hidden_dim)

        # Graph attention layers
        self.gat_layers = nn.ModuleList()
        for _ in range(num_layers):
            self.gat_layers.append(GraphAttention(
                in_features=hidden_dim,
                out_features=self.out_features_per_head,
                n_heads=n_heads,
                dropout=dropout
            ))

        # Sequence generation
        self.rnn_cell = nn.GRUCell(hidden_dim, hidden_dim)
        self.sequence_projection = nn.Linear(hidden_dim, amino_acid_vocab_size)

        # Secondary structure prediction
        self.ss_projection = nn.Linear(hidden_dim, 9)  # 9 SS types

        # Adjacency matrix generation
        self.edge_predictor = self.create_enhanced_edge_predictor()

        self.dropout = nn.Dropout(dropout)

    def create_enhanced_edge_predictor(self):
        """Enhanced edge predictor with more layers"""
        return nn.Sequential(
            nn.Linear(self.hidden_dim * 2, self.hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(self.hidden_dim, self.hidden_dim // 2),
            nn.ReLU(),
            nn.Linear(self.hidden_dim // 2, 1),
            nn.Sigmoid()
        )

    def compute_advanced_loss(self, predictions, targets):
        """Advanced loss computation with structural constraints and SS prediction"""
        # Existing sequence loss
        seq_logits = predictions['sequence_logits']
        target_seq = targets['sequence']
        batch_size, seq_len, vocab_size = seq_logits.size()
        seq_logits_flat = seq_logits.view(-1, vocab_size)
        target_seq_flat = target_seq.view(-1)
        sequence_loss = F.cross_entropy(seq_logits_flat, target_seq_flat)

        # Enhanced adjacency matrix loss with constraints
        pred_adj = predictions['adjacency_matrix']
        target_adj = targets['adjacency_matrix']

        # 1. Basic contact loss (excluding diagonal)
        mask = 1 - torch.eye(pred_adj.size(1), device=pred_adj.device).unsqueeze(0)
        basic_contact_loss = F.binary_cross_entropy(
            pred_adj * mask,
            target_adj * mask,
            reduction='mean'
        )

        # 2. Sequential distance constraint (consecutive residues ~3.8Å)
        sequential_distance_loss = self.compute_sequential_distance_loss(pred_adj)

        # 3. Symmetry loss
        symmetry_loss = self.compute_symmetry_loss(pred_adj)

        # 4. Distance-based contact classification loss
        distance_loss = self.compute_distance_based_loss(pred_adj, target_adj)

        # Combined adjacency loss with weights
        adjacency_loss = (
                1.0 * basic_contact_loss +
                0.3 * sequential_distance_loss +
                0.3 * symmetry_loss +
                0.4 * distance_loss
        )

        # 5. Secondary structure loss
        ss_logits = predictions['ss_logits']
        # Extract SS labels from node features (positions 7+22 = 29 onwards)
        ss_features = targets['node_features'][:, :, 7+self.amino_acid_vocab_size:]
        ss_labels = torch.argmax(ss_features, dim=-1)
        ss_logits_flat = ss_logits.view(-1, 9)
        ss_labels_flat = ss_labels.view(-1)
        ss_loss = F.cross_entropy(ss_logits_flat, ss_labels_flat)

        # Combined loss with weights
        combined_loss = sequence_loss + adjacency_loss + 0.5 * ss_loss

        return {
            'combined_loss': combined_loss,
            'sequence_loss': sequence_loss,
            'adjacency_loss': adjacency_loss,
            'ss_loss': ss_loss,
            'contact_components': {
                'basic_contact': basic_contact_loss,
                'sequential_distance': sequential_distance_loss,
                'symmetry': symmetry_loss,
                'distance_classification': distance_loss
            }
        }

    def compute_sequential_distance_loss(self, pred_adj):
        """Ensure sequential residues have appropriate distance (~3.8Å)"""
        # Get diagonal bands (distances 1 and 2)
        batch_size, N, _ = pred_adj.size()

        # Distance 1 (adjacent residues)
        diag1 = torch.diagonal(pred_adj, offset=1, dim1=1, dim2=2)
        # Expected probability for distance of 3.8Å in contact maps
        target_prob1 = torch.ones_like(diag1) * 0.9
        loss1 = F.mse_loss(diag1, target_prob1)

        # Distance 2 (residues separated by one)
        diag2 = torch.diagonal(pred_adj, offset=2, dim1=1, dim2=2)
        target_prob2 = torch.ones_like(diag2) * 0.7
        loss2 = F.mse_loss(diag2, target_prob2)

        return (loss1 + loss2) / 2

    def compute_symmetry_loss(self, pred_adj):
        """Enforce symmetry in contact maps"""
        symmetric_diff = torch.abs(pred_adj - pred_adj.transpose(1, 2))
        return torch.mean(symmetric_diff)

    def compute_distance_based_loss(self, pred_adj, target_adj):
        """Classify contacts by distance ranges"""
        batch_size, N, _ = pred_adj.size()

        # Create masks for different distance ranges
        # Local (|i-j| <= 5)
        local_mask = torch.zeros_like(pred_adj)
        for d in range(1, 6):
            local_mask += torch.eye(N, device=pred_adj.device).roll(shifts=d, dims=0).unsqueeze(0)
            local_mask += torch.eye(N, device=pred_adj.device).roll(shifts=-d, dims=0).unsqueeze(0)

        # Medium-range (5 < |i-j| <= 20)
        medium_mask = torch.zeros_like(pred_adj)
        for d in range(6, 21):
            medium_mask += torch.eye(N, device=pred_adj.device).roll(shifts=d, dims=0).unsqueeze(0)
            medium_mask += torch.eye(N, device=pred_adj.device).roll(shifts=-d, dims=0).unsqueeze(0)

        # Long-range (|i-j| > 20)
        long_mask = torch.ones_like(pred_adj) - local_mask - medium_mask
        # Remove diagonal
        long_mask = long_mask * (1 - torch.eye(N, device=pred_adj.device).unsqueeze(0))

        # Compute losses for each range
        local_loss = F.binary_cross_entropy(pred_adj * local_mask, target_adj * local_mask, reduction='sum') / (local_mask.sum() + 1e-8)
        medium_loss = F.binary_cross_entropy(pred_adj * medium_mask, target_adj * medium_mask, reduction='sum') / (medium_mask.sum() + 1e-8)
        long_loss = F.binary_cross_entropy(pred_adj * long_mask, target_adj * long_mask, reduction='sum') / (long_mask.sum() + 1e-8)

        # Weight the losses differently
        return 0.3 * local_loss + 0.3 * medium_loss + 0.4 * long_loss

    def _process_graph(self, node_features, adjacency_matrix):
        """Process the input graph with graph attention layers"""
        h = self.node_embedding(node_features)
        for gat_layer in self.gat_layers:
            h_residual = h  # Save input
            h = gat_layer(h, adjacency_matrix)
            h = F.elu(h + h_residual)  # Add input back to output
            h = self.dropout(h)

        return h

    def _generate_adjacency(self, node_embeddings):
        """Generate adjacency matrix from node embeddings"""
        batch_size, num_nodes, _ = node_embeddings.size()

        # Create all pairwise combinations of node embeddings
        node_i = node_embeddings.unsqueeze(2).repeat(1, 1, num_nodes, 1)  # [B, N, N, H]
        node_j = node_embeddings.unsqueeze(1).repeat(1, num_nodes, 1, 1)  # [B, N, N, H]

        # Concatenate node pairs
        node_pairs = torch.cat([node_i, node_j], dim=-1)  # [B, N, N, 2H]

        # Reshape for passing through edge predictor
        flat_pairs = node_pairs.view(-1, 2 * self.hidden_dim)  # [B*N*N, 2H]

        # Predict edges
        edge_scores = self.edge_predictor(flat_pairs).view(batch_size, num_nodes, num_nodes)  # [B, N, N]

        # Ensure symmetry (for undirected graphs)
        edge_scores = (edge_scores + edge_scores.transpose(1, 2)) / 2

        return edge_scores

    def forward(self, node_features, adjacency_matrix, target_sequences=None, target_adjacency=None, max_length=50):
        """
        Forward pass with dual outputs (sequence and adjacency matrix)

        Args:
            node_features: [batch_size, num_nodes, node_feature_dim]
            adjacency_matrix: [batch_size, num_nodes, num_nodes]
            target_sequences: [batch_size, seq_length] (for training)
            target_adjacency: [batch_size, num_nodes, num_nodes] (for training)
            max_length: Maximum sequence length for generation

        Returns:
            Dictionary containing:
                - 'sequence_logits': Predicted sequence logits
                - 'adjacency_matrix': Predicted adjacency matrix
                - 'ss_logits': Predicted secondary structure logits
                - Or generated sequence and adjacency matrix during inference
        """
        batch_size, num_nodes = node_features.size(0), node_features.size(1)

        # Process graph
        node_embeddings = self._process_graph(node_features, adjacency_matrix)  # [B, N, H]

        # Generate adjacency matrix
        predicted_adjacency = self._generate_adjacency(node_embeddings)  # [B, N, N]

        # Predict secondary structure
        ss_logits = self.ss_projection(node_embeddings)  # [B, N, 9]

        # Initialize RNN for sequence generation
        graph_embedding = torch.mean(node_embeddings, dim=1)  # [B, H]
        h_t = graph_embedding  # Initial hidden state

        if target_sequences is not None:
            # Training mode
            seq_length = target_sequences.size(1)
            sequence_logits = torch.zeros(batch_size, seq_length, self.amino_acid_vocab_size,
                                          device=node_features.device)

            x_t = torch.zeros(batch_size, self.hidden_dim, device=node_features.device)

            for t in range(seq_length):
                h_t = self.rnn_cell(x_t, h_t)  # [B, H]
                sequence_logits[:, t, :] = self.sequence_projection(h_t)  # [B, vocab_size]

                if t < seq_length - 1:
                    # Create a full feature vector for the amino acid
                    amino_acid_feature = torch.zeros(batch_size, self.node_features, device=node_features.device)

                    # Set the amino acid one-hot encoding (positions 7-28)
                    aa_onehot = F.one_hot(target_sequences[:, t], num_classes=self.amino_acid_vocab_size).float()
                    amino_acid_feature[:, 7:7+self.amino_acid_vocab_size] = aa_onehot

                    # Pass through node embedding
                    x_t = self.node_embedding(amino_acid_feature)  # [B, H]

            return {
                'sequence_logits': sequence_logits,
                'adjacency_matrix': predicted_adjacency,
                'ss_logits': ss_logits
            }

        else:
            # Generation mode
            generated_sequences = torch.zeros(batch_size, max_length,
                                              dtype=torch.long,
                                              device=node_features.device)
            x_t = torch.zeros(batch_size, self.hidden_dim,
                              device=node_features.device)

            # Get the full SS predictions upfront
            predicted_ss_full = torch.argmax(ss_logits, dim=-1)  # [B, N]

            for t in range(max_length):
                h_t = self.rnn_cell(x_t, h_t)
                output = self.sequence_projection(h_t)
                prob = F.softmax(output, dim=-1)
                next_aa = torch.multinomial(prob, 1).squeeze(-1)
                generated_sequences[:, t] = next_aa

                # Create a full feature vector for the amino acid
                amino_acid_feature = torch.zeros(batch_size, self.node_features, device=node_features.device)

                # Set the amino acid one-hot encoding (positions 7-28)
                aa_onehot = F.one_hot(next_aa, num_classes=self.amino_acid_vocab_size).float()
                amino_acid_feature[:, 7:7+self.amino_acid_vocab_size] = aa_onehot

                # Add SS prediction if within bounds - simple feedback approach
                if t < predicted_ss_full.size(1):
                    # Get SS prediction for current position
                    ss_onehot = F.one_hot(predicted_ss_full[:, t], num_classes=9).float()
                    # Put SS in positions 29-37
                    amino_acid_feature[:, 7+self.amino_acid_vocab_size:] = ss_onehot

                # Pass through node embedding with both AA and SS info
                x_t = self.node_embedding(amino_acid_feature)  # [B, H]

            return {
                'generated_sequence': generated_sequences,
                'adjacency_matrix': predicted_adjacency,
                'predicted_ss': predicted_ss_full  # [B, N]
            }



# Data Loading and preprocessing of graphs, dataloaders

In [ ]:

def load_protein_graph_data(parent_folder, max_proteins=None, chunk_length=chunk_length):
    """
    Load protein NetworkX graph data with detailed diagnostics
    """
    random.seed(42)

    # List to store graphs and their associated amino acid sequences
    protein_graphs = []
    protein_sequences = []

    # Diagnostic counters
    loaded_proteins = 0
    failed_proteins = 0

    folders = [name for name in os.listdir(parent_folder)
               if os.path.isdir(os.path.join(parent_folder, name))]

    found_folders = len(folders)
    print(f"Found {found_folders} protein folders")

    # Load only max_proteins if specified
    if max_proteins:
        folders = folders[:max_proteins]
        print(f"Limited to {len(folders)} folders due to max_proteins setting")

    for folder in folders:
        # Look for graph files in the folder
        folder_path = os.path.join(parent_folder, folder)
        graph_files = [f for f in os.listdir(folder_path) if f.endswith('_graph.pkl')]

        if graph_files:
            graph_file = os.path.join(folder_path, graph_files[0])
            try:
                with open(graph_file, 'rb') as f:
                    graph = pkl.load(f)

                    # Extract amino acid sequence
                    aa_seq = []
                    sorted_nodes = sorted(graph.nodes(), key=lambda x:
                    int(graph.nodes[x]['residue_number'])
                    if 'residue_number' in graph.nodes[x] else 0)

                    for node in sorted_nodes:
                        if 'residue_name' in graph.nodes[node]:
                            aa_seq.append(graph.nodes[node]['residue_name'])
                        else:
                            aa_seq.append('X')

                    protein_graphs.append(graph)
                    protein_sequences.append(aa_seq)
                    loaded_proteins += 1
            except Exception as e:
                failed_proteins += 1
                print(f"Error loading {graph_file}: {e}")

    print(f"Successfully loaded {loaded_proteins} protein graphs, {failed_proteins} failed")

    # Create smaller subgraphs
    subgraphs = []
    subsequences = []

    # Diagnostic counters for subsequence generation
    total_possible_subsequences = 0
    actual_generated_subsequences = 0
    proteins_with_no_subsequences = 0

    for i, (graph, sequence) in enumerate(zip(protein_graphs, protein_sequences)):
        # Extract nodes by residue number ranges
        sorted_nodes = sorted(graph.nodes(), key=lambda x:
        int(graph.nodes[x]['residue_number'])
        if 'residue_number' in graph.nodes[x] else 0)

        # Count theoretical maximum for this protein
        seq_length = len(sorted_nodes)
        max_subseqs_this_protein = seq_length  # With wrapping, we can generate as many subsequences as residues
        total_possible_subsequences += max_subseqs_this_protein

        subseqs_this_protein = 0

        # Create overlapping subsequences with wrapping
        for start_idx in range(0, seq_length, step_size):
            # Generate indices for the subsequence with wrapping
            indices = []
            for j in range(chunk_length):
                # Wrap around if we exceed the length
                wrapped_idx = (start_idx + j) % seq_length
                indices.append(wrapped_idx)

            # Get the corresponding nodes
            node_subset = [sorted_nodes[idx] for idx in indices]
            # Get the corresponding sequence
            aa_subset = [sequence[idx] for idx in indices]

            if node_subset:
                try:
                    subgraph = graph.subgraph(node_subset)
                    if len(subgraph) > 0:
                        subgraphs.append(subgraph)
                        subsequences.append(aa_subset)
                        subseqs_this_protein += 1
                        actual_generated_subsequences += 1
                except Exception as e:
                    print(f"Error creating subgraph: {e}")

        if subseqs_this_protein == 0:
            proteins_with_no_subsequences += 1

    print(f"\nSubsequence Generation Statistics:")
    print(f"Total possible subsequences: {total_possible_subsequences}")
    print(f"Actually generated subsequences: {actual_generated_subsequences}")
    print(f"Proteins with no subsequences: {proteins_with_no_subsequences}")

    # If no subgraphs were created, use the full graphs
    if not subgraphs:
        print("Warning: Could not create subgraphs. Using full graphs instead.")
        subgraphs = protein_graphs
        subsequences = protein_sequences

    print(f"Final count: {len(subgraphs)} protein subgraphs for training")

    # Shuffle the data
    if len(subgraphs) > 0:
        combined = list(zip(subgraphs, subsequences))
        random.shuffle(combined)
        subgraphs, subsequences = zip(*combined)

    # Limit size if needed
    if max_proteins and len(subgraphs) > subgraph_limit:
        print(f"Limiting from {len(subgraphs)} to {subgraph_limit} due to max_proteins /subgraph limit")
        subgraphs = subgraphs[:subgraph_limit]
        subsequences = subsequences[:subgraph_limit]

    return protein_graphs, protein_sequences, list(subgraphs), list(subsequences)

def prepare_graph_data_for_training(subgraphs, subsequences, unique_aa):
    """
    Prepare graph data for GRAN model training with enhanced node features

    Args:
        subgraphs: List of NetworkX subgraphs
        subsequences: List of amino acid sequences corresponding to the subgraphs
        unique_aa: Set of unique amino acids to use for one-hot encoding

    Returns:
        aa_sequences_tensor: Tensor of amino acid sequences (targets)
        adjacency_tensors: Tensor of graph adjacency matrices
        node_features_tensor: Tensor of node features
    """
    # Secondary structure mapping
    SS_TO_INDEX = {
        'E': 0,  # Extended (beta sheets)
        '-': 1,  # Coil
        'T': 2,  # Turn
        'S': 3,  # Bend
        'G': 4,  # 3-10 helix
        'H': 5,  # Alpha helix
        'B': 6,  # Bridge
        'I': 7,  # Pi helix
        '?': 8   # Unknown
    }

    # Prepare amino acid sequences (these will be our targets)
    aa_sequences = []
    for seq in subsequences:
        # Convert amino acid sequence to indices
        aa_indices = []
        for aa in seq:
            if aa in unique_aa:
                aa_indices.append(list(unique_aa).index(aa))
            else:
                # Handle unknown amino acids
                aa_indices.append(list(unique_aa).index('X') if 'X' in unique_aa else 0)
        aa_sequences.append(aa_indices)

    # Prepare adjacency matrices and node features
    adjacency_matrices = []
    node_features = []

    for i, graph in enumerate(subgraphs):
        # Get number of nodes
        num_nodes = len(graph)

        # Skip empty graphs
        if num_nodes == 0:
            continue

        # Create adjacency matrix
        adj_matrix = nx.to_numpy_array(graph)
        adjacency_matrices.append(adj_matrix)

        # Extract ordered nodes
        sorted_nodes = sorted(graph.nodes(), key=lambda x:
        int(graph.nodes[x]['residue_number'])
        if 'residue_number' in graph.nodes[x] else 0)

        # Create enhanced features: Meiler (7) + AA one-hot (22) + SS one-hot (9) = 38
        FEATURE_DIM = 7 + len(unique_aa) + 9
        features = np.zeros((num_nodes, FEATURE_DIM))

        for j, node in enumerate(sorted_nodes):
            # Meiler features (first 7 dimensions)
            if 'meiler' in graph.nodes[node]:
                meiler_values = graph.nodes[node]['meiler']
                # Extract meiler dimensions - handle different formats
                if isinstance(meiler_values, dict):
                    # Format: {'dim_1': 1.28, 'dim_2': 0.05, ...}
                    features[j, 0] = meiler_values.get('dim_1', 0.0)
                    features[j, 1] = meiler_values.get('dim_2', 0.0)
                    features[j, 2] = meiler_values.get('dim_3', 0.0)
                    features[j, 3] = meiler_values.get('dim_4', 0.0)
                    features[j, 4] = meiler_values.get('dim_5', 0.0)
                    features[j, 5] = meiler_values.get('dim_6', 0.0)
                    features[j, 6] = meiler_values.get('dim_7', 0.0)
                elif isinstance(meiler_values, list) and len(meiler_values) >= 7:
                    # Format: [1.28, 0.05, 1.00, 0.31, 6.11, 0.42, 0.23]
                    features[j, :7] = meiler_values[:7]

            # Amino acid one-hot (dimensions 7-28)
            if 'residue_name' in graph.nodes[node]:
                aa = graph.nodes[node]['residue_name']
                if aa in unique_aa:
                    aa_idx = list(unique_aa).index(aa)
                    features[j, 7 + aa_idx] = 1.0
                elif 'X' in unique_aa:
                    aa_idx = list(unique_aa).index('X')
                    features[j, 7 + aa_idx] = 1.0
                else:
                    features[j, 7] = 1.0

            # Secondary structure one-hot (dimensions 29-37)
            if 'secondary_structure' in graph.nodes[node]:
                ss = graph.nodes[node]['secondary_structure']
                if ss in SS_TO_INDEX:
                    ss_idx = SS_TO_INDEX[ss]
                    features[j, 7 + len(unique_aa) + ss_idx] = 1.0
                else:
                    # Unknown SS -> use '?' index
                    features[j, 7 + len(unique_aa) + SS_TO_INDEX['?']] = 1.0

        node_features.append(features)

    # Check if we have data
    if not adjacency_matrices or not node_features:
        raise ValueError("No valid graphs found after processing")

    # Convert to tensors
    aa_sequences_tensor = [torch.tensor(seq, dtype=torch.long) for seq in aa_sequences if seq]
    adjacency_tensors = [torch.tensor(adj, dtype=torch.float32) for adj in adjacency_matrices]
    node_features_tensor = [torch.tensor(nf, dtype=torch.float32) for nf in node_features]

    return aa_sequences_tensor, adjacency_tensors, node_features_tensor

def collate_batch(batch):
    aa_seqs, adjacency_matrices, node_feats = zip(*batch)
    # Stack directly without padding since all should be size 50
    return torch.stack(aa_seqs), torch.stack(adjacency_matrices), torch.stack(node_feats)


def create_dataloader(aa_sequences, adjacency_matrices, node_features, batch_size=batch_size, shuffle=True):
    """
    Create a dataloader from the prepared data
    """
    dataset = list(zip(aa_sequences, adjacency_matrices, node_features))
    dataloader = torch.utils.data.DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        collate_fn=collate_batch
    )
    return dataloader



# Model loading if it already exists

In [ ]:
def load_model(model_path, model, device):
    """
    Load a saved model - handles both formats

    Args:
        model_path: Path to the saved model
        model: Model instance to load the weights into
        device: Device to load the model to

    Returns:
        The loaded model
    """
    if not os.path.exists(model_path):
        print(f"No checkpoint found at {model_path}")
        return model

    try:
        # Try to load as a dictionary first
        checkpoint = torch.load(model_path, map_location=device)
        if isinstance(checkpoint, dict) and 'model_state_dict' in checkpoint:
            model.load_state_dict(checkpoint['model_state_dict'])
        else:
            # Assume it's just the state dict
            model.load_state_dict(checkpoint)
        print(f"Successfully loaded model from {model_path}")
    except Exception as e:
        print(f"Error loading model: {e}")

    return model

# Load data -
some of this is just sanity checks and global things needed later, adjacency matrices, AA sequences

In [ ]:

# Load protein graph data directly
try:
    full_graphs, full_sequences, subgraphs, subsequences = load_protein_graph_data(
        parent_folder, max_proteins, chunk_length
    )
except Exception as e:
    print(f"Error loading graph data: {e}")
    print("Current directory contains:", os.listdir())



# First graph debug
print("First few nodes of first graph:")
first_graph = full_graphs[0]
for i, node in enumerate(sorted(first_graph.nodes())[:5]):
    print(f"Node {node} attributes: {first_graph.nodes[node]}")

# Define a standard set of amino acids (all 20 standard ones)
STANDARD_AA = ['ALA', 'ARG', 'ASN', 'ASP', 'CYS', 'GLN', 'GLU', 'GLY', 'HIS', 'ILE',
               'LEU', 'LYS', 'MET', 'PHE', 'PRO', 'SER', 'THR', 'TRP', 'TYR', 'VAL', 'X']

# Get unique amino acids from sequences but ensure we have at least the standard 20
UNIQUE_AA = set()
for seq in full_sequences:
    UNIQUE_AA.update(seq)

# Merge with standard AAs
UNIQUE_AA = sorted(list(set(UNIQUE_AA).union(set(STANDARD_AA))))
print(f"Unique amino acids: {len(UNIQUE_AA)}")
print(f"Amino acids found: {', '.join(UNIQUE_AA)}")

# Prepare data for training using graph data
aa_sequences, adjacency_matrices, node_features = prepare_graph_data_for_training(
    subgraphs, subsequences, UNIQUE_AA
)

print(f"Prepared data: {len(aa_sequences)} sequences, {len(adjacency_matrices)} adjacency matrices")


# Model training
includes lots of logging and debugging code :-(

In [ ]:
def train_dual_output_model(model, train_loader, val_loader, num_epochs=num_epochs, lr=lr, device='cpu',
                            patience=patience, min_epochs=min_epochs, checkpoint_path='best_model.pt'):
    """
    Train the dual output GRAN model with improved early stopping and checkpointing

    Args:
        model: The model to train
        train_loader: Training data loader
        val_loader: Validation data loader
        num_epochs: Maximum number of epochs to train
        lr: Learning rate
        device: Device to train on (cpu/gpu)
        patience: Number of epochs to wait for improvement before stopping
        min_epochs: Minimum number of epochs to train regardless of early stopping
        checkpoint_path: Path to save the best model

    Returns:
        Dictionary of training and validation losses
    """

    run = wandb.init(
        # Set the wandb entity where your project will be logged (generally your team name).
        entity="ampc22-bath-university",
        # Set the wandb project where this run will be logged.
        project="GranTest",
        # Track hyperparameters and run metadata.
        config={
            "chunk_length": chunk_length,
            "learning_rate": lr,
            "architecture": "GRAN_inspired",
            "dataset": "Protein",
            "epochs": num_epochs,
        },
    )

    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    # Optional: Add a learning rate scheduler to reduce LR when loss plateaus
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min',
                                                           factor=0.5, patience=patience//2)

    train_losses = {'combined': [], 'sequence': [], 'adjacency': []}
    val_losses = {'combined': [], 'sequence': [], 'adjacency': []}

    # Variables for early stopping and checkpointing
    best_val_loss = float('inf')
    early_stop_counter = 0
    best_epoch = 0

    for epoch in range(num_epochs):
        # Training
        model.train()
        epoch_train_losses = {'combined': 0, 'sequence': 0, 'adjacency': 0}
        batch_count = 0

        for i, (aa_seqs, adjacency_matrices, node_feats) in enumerate(train_loader):
            # Skip empty batches
            if len(aa_seqs) == 0:
                continue

            # Move data to device
            aa_seqs = aa_seqs.to(device)
            adjacency_matrices = adjacency_matrices.to(device)
            node_feats = node_feats.to(device)

            try:
                # Forward pass
                predictions = model(
                    node_feats,
                    adjacency_matrices,
                    target_sequences=aa_seqs,
                    target_adjacency=adjacency_matrices
                )

                # Compute loss
                targets = {
                    'sequence': aa_seqs,
                    'adjacency_matrix': adjacency_matrices
                }

                losses = model.compute_loss(predictions, targets)

                # Backward pass and optimization
                optimizer.zero_grad()
                losses['combined_loss'].backward()
                optimizer.step()

                # Record losses
                epoch_train_losses['combined'] += losses['combined_loss'].item()
                epoch_train_losses['sequence'] += losses['sequence_loss'].item()
                epoch_train_losses['adjacency'] += losses['adjacency_loss'].item()
                batch_count += 1

                if i == 0:
                    print(f"Epoch {epoch+1}, Batch {i}: Combined Loss: {losses['combined_loss'].item():.4f}, "
                          f"Seq Loss: {losses['sequence_loss'].item():.4f}, "
                          f"Adj Loss: {losses['adjacency_loss'].item():.4f}")

            except Exception as e:
                print(f"Error in batch {i}: {e}")
                continue

        # Average train losses
        for key in epoch_train_losses:
            if batch_count > 0:
                epoch_train_losses[key] /= batch_count
                train_losses[key].append(epoch_train_losses[key])

        # Validation
        model.eval()
        epoch_val_losses = {'combined': 0, 'sequence': 0, 'adjacency': 0}
        val_batch_count = 0

        with torch.no_grad():
            for aa_seqs, adjacency_matrices, node_feats in val_loader:
                if len(aa_seqs) == 0:
                    continue

                aa_seqs = aa_seqs.to(device)
                adjacency_matrices = adjacency_matrices.to(device)
                node_feats = node_feats.to(device)

                try:
                    # Forward pass
                    predictions = model(
                        node_feats,
                        adjacency_matrices,
                        target_sequences=aa_seqs,
                        target_adjacency=adjacency_matrices
                    )

                    # Compute loss
                    targets = {
                        'sequence': aa_seqs,
                        'adjacency_matrix': adjacency_matrices
                    }

                    losses = model.compute_loss(predictions, targets)

                    # Record losses
                    epoch_val_losses['combined'] += losses['combined_loss'].item()
                    epoch_val_losses['sequence'] += losses['sequence_loss'].item()
                    epoch_val_losses['adjacency'] += losses['adjacency_loss'].item()
                    val_batch_count += 1

                except Exception as e:
                    print(f"Error in validation: {e}")
                    continue

        # Average validation losses
        for key in epoch_val_losses:
            if val_batch_count > 0:
                epoch_val_losses[key] /= val_batch_count
                val_losses[key].append(epoch_val_losses[key])

        # Update learning rate scheduler
        current_val_loss = epoch_val_losses['combined']
        scheduler.step(current_val_loss)

        # Check if this is the best model so far
        if current_val_loss < best_val_loss:
            best_val_loss = current_val_loss
            best_epoch = epoch
            early_stop_counter = 0

            # Save the best model - SIMPLIFIED VERSION
            # Just save the model state dict directly
            torch.save(model.state_dict(), checkpoint_path)
            print(f"Checkpoint saved at epoch {epoch+1} with validation loss: {best_val_loss:.4f}")
        else:
            early_stop_counter += 1

        # Print epoch summary
        print(f"Epoch {epoch+1}/{num_epochs} - "
              f"Train: Combined {epoch_train_losses['combined']:.4f}, "
              f"Seq {epoch_train_losses['sequence']:.4f}, "
              f"Adj {epoch_train_losses['adjacency']:.4f} | "
              f"Val: Combined {epoch_val_losses['combined']:.4f}, "
              f"Seq {epoch_val_losses['sequence']:.4f}, "
              f"Adj {epoch_val_losses['adjacency']:.4f}")

        wandb.log({
            "train_combined_loss": epoch_train_losses['combined'],
            "train_sequence_loss": epoch_train_losses['sequence'],
            "train_adjacency_loss": epoch_train_losses['adjacency'],
            "val_combined_loss": epoch_val_losses['combined'],
            "val_sequence_loss": epoch_val_losses['sequence'],
            "val_adjacency_loss": epoch_val_losses['adjacency'],
            "epoch": epoch + 1
        })
        # Early stopping check (but only after minimum epochs)
        if epoch >= min_epochs and early_stop_counter >= patience:
            print(f"Early stopping triggered after {epoch+1} epochs. Best model was at epoch {best_epoch+1}.")
            # Load the best model before returning
            model.load_state_dict(torch.load(checkpoint_path))
            break

    # If training completed without early stopping, load the best model
    if epoch == num_epochs - 1:
        print(f"Training completed. Loading best model from epoch {best_epoch+1}.")
        model.load_state_dict(torch.load(checkpoint_path))

    return train_losses, val_losses



In [ ]:
# Enhanced training function with detailed loss tracking and wandb integration
def train_dual_output_model_enhanced(model, train_loader, val_loader, num_epochs=num_epochs, lr=lr, device='cpu',
                                     patience=30, min_epochs=1, checkpoint_path='best_model.pt'):
    """Enhanced training with detailed loss tracking and wandb logging"""


    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=patience//2)

    # Enhanced loss tracking
    train_losses = {
        'combined': [], 'sequence': [], 'adjacency': [],
        'contact_components': {
            'basic_contact': [], 'sequential_distance': [],
            'symmetry': [], 'distance_classification': []
        }
    }
    val_losses = {
        'combined': [], 'sequence': [], 'adjacency': [],
        'contact_components': {
            'basic_contact': [], 'sequential_distance': [],
            'symmetry': [], 'distance_classification': []
        }
    }

    best_val_loss = float('inf')
    early_stop_counter = 0
    best_epoch = 0

    for epoch in range(num_epochs):
        # Training
        model.train()
        epoch_train_losses = {
            'combined': 0, 'sequence': 0, 'adjacency': 0,
            'contact_components': {k: 0 for k in train_losses['contact_components']}
        }
        batch_count = 0

        for i, (aa_seqs, adjacency_matrices, node_feats) in enumerate(train_loader):
            if len(aa_seqs) == 0:
                continue

            aa_seqs = aa_seqs.to(device)
            adjacency_matrices = adjacency_matrices.to(device)
            node_feats = node_feats.to(device)

            try:
                # Forward pass
                predictions = model(
                    node_feats,
                    adjacency_matrices,
                    target_sequences=aa_seqs,
                    target_adjacency=adjacency_matrices
                )

                # Compute advanced loss
                targets = {
                    'sequence': aa_seqs,
                    'adjacency_matrix': adjacency_matrices,
                    'node_features': node_feats  # Add this line
                }


                losses = model.compute_advanced_loss(predictions, targets)

                # Backward pass
                optimizer.zero_grad()
                losses['combined_loss'].backward()
                optimizer.step()

                # Record losses
                epoch_train_losses['combined'] += losses['combined_loss'].item()
                epoch_train_losses['sequence'] += losses['sequence_loss'].item()
                epoch_train_losses['adjacency'] += losses['adjacency_loss'].item()

                for k, v in losses['contact_components'].items():
                    epoch_train_losses['contact_components'][k] += v.item()

                batch_count += 1

                if i == 0:
                    print(f"Epoch {epoch+1}, Batch {i}:")
                    print(f"  Combined Loss: {losses['combined_loss'].item():.4f}")
                    print(f"  Sequence Loss: {losses['sequence_loss'].item():.4f}")
                    print(f"  Adjacency Loss: {losses['adjacency_loss'].item():.4f}")
                    print(f"  Contact Components:")
                    for k, v in losses['contact_components'].items():
                        print(f"    {k}: {v.item():.4f}")

            except Exception as e:
                print(f"Error in batch {i}: {e}")
                continue

        # Average train losses
        for key in epoch_train_losses:
            if key != 'contact_components':
                if batch_count > 0:
                    epoch_train_losses[key] /= batch_count
                    train_losses[key].append(epoch_train_losses[key])
            else:
                for comp_key in epoch_train_losses['contact_components']:
                    if batch_count > 0:
                        epoch_train_losses['contact_components'][comp_key] /= batch_count
                        train_losses['contact_components'][comp_key].append(epoch_train_losses['contact_components'][comp_key])

        # Validation
        model.eval()
        epoch_val_losses = {
            'combined': 0, 'sequence': 0, 'adjacency': 0,
            'contact_components': {k: 0 for k in train_losses['contact_components']}
        }
        val_batch_count = 0

        with torch.no_grad():
            for aa_seqs, adjacency_matrices, node_feats in val_loader:
                if len(aa_seqs) == 0:
                    continue

                aa_seqs = aa_seqs.to(device)
                adjacency_matrices = adjacency_matrices.to(device)
                node_feats = node_feats.to(device)

                try:
                    # Forward pass
                    predictions = model(
                        node_feats,
                        adjacency_matrices,
                        target_sequences=aa_seqs,
                        target_adjacency=adjacency_matrices
                    )

                    # Compute loss
                    targets = {
                        'sequence': aa_seqs,
                        'adjacency_matrix': adjacency_matrices,
                        'node_features': node_feats  # Add this line
                    }


                    losses = model.compute_advanced_loss(predictions, targets)

                    # Record losses
                    epoch_val_losses['combined'] += losses['combined_loss'].item()
                    epoch_val_losses['sequence'] += losses['sequence_loss'].item()
                    epoch_val_losses['adjacency'] += losses['adjacency_loss'].item()

                    for k, v in losses['contact_components'].items():
                        epoch_val_losses['contact_components'][k] += v.item()

                    val_batch_count += 1

                except Exception as e:
                    print(f"Error in validation: {e}")
                    continue

        # Average validation losses
        for key in epoch_val_losses:
            if key != 'contact_components':
                if val_batch_count > 0:
                    epoch_val_losses[key] /= val_batch_count
                    val_losses[key].append(epoch_val_losses[key])
            else:
                for comp_key in epoch_val_losses['contact_components']:
                    if val_batch_count > 0:
                        epoch_val_losses['contact_components'][comp_key] /= val_batch_count
                        val_losses['contact_components'][comp_key].append(epoch_val_losses['contact_components'][comp_key])

        # Update learning rate and check early stopping
        current_val_loss = epoch_val_losses['combined']
        scheduler.step(current_val_loss)

        if current_val_loss < best_val_loss:
            best_val_loss = current_val_loss
            best_epoch = epoch
            early_stop_counter = 0
            torch.save(model.state_dict(), checkpoint_path)
            print(f"Checkpoint saved at epoch {epoch+1} with validation loss: {best_val_loss:.4f}")
        else:
            early_stop_counter += 1

        # Print detailed epoch summary
        print(f"Epoch {epoch+1}/{num_epochs} Summary:")
        print(f"Train - Combined: {epoch_train_losses['combined']:.4f}, Seq: {epoch_train_losses['sequence']:.4f}, Adj: {epoch_train_losses['adjacency']:.4f}")
        print(f"Val - Combined: {epoch_val_losses['combined']:.4f}, Seq: {epoch_val_losses['sequence']:.4f}, Adj: {epoch_val_losses['adjacency']:.4f}")
        print("Contact Component Losses:")
        for k in epoch_train_losses['contact_components']:
            print(f"  {k} - Train: {epoch_train_losses['contact_components'][k]:.4f}, Val: {epoch_val_losses['contact_components'][k]:.4f}")

        # Log to wandb


        # Add contact component losses to wandb logs
        for k in epoch_train_losses['contact_components']:
           print("")

        # Early stopping check
        if epoch >= min_epochs and early_stop_counter >= patience:
            print(f"Early stopping triggered after {epoch+1} epochs. Best model was at epoch {best_epoch+1}.")
            model.load_state_dict(torch.load(checkpoint_path))
            break

    return train_losses, val_losses

In [ ]:
# Enhanced visualization function for loss components
def plot_train_results_enhanced(train_losses, val_losses):
    """Enhanced visualization for all loss components"""
    import matplotlib.pyplot as plt

    # Create a comprehensive figure with subplots
    fig = plt.figure(figsize=(20, 15))
    gs = fig.add_gridspec(3, 3, hspace=0.3, wspace=0.3)

    # Main losses (larger plots)
    ax1 = fig.add_subplot(gs[0, :])
    ax1.plot(train_losses['combined'], label='Train Combined', linewidth=2)
    ax1.plot(val_losses['combined'], label='Val Combined', linewidth=2)
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Loss')
    ax1.set_title('Combined Loss')
    ax1.legend()
    ax1.grid(True, alpha=0.3)

    # Sequence and Adjacency losses
    ax2 = fig.add_subplot(gs[1, 0])
    ax2.plot(train_losses['sequence'], label='Train', linewidth=2)
    ax2.plot(val_losses['sequence'], label='Val', linewidth=2)
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Loss')
    ax2.set_title('Sequence Loss')
    ax2.legend()
    ax2.grid(True, alpha=0.3)

    ax3 = fig.add_subplot(gs[1, 1])
    ax3.plot(train_losses['adjacency'], label='Train', linewidth=2)
    ax3.plot(val_losses['adjacency'], label='Val', linewidth=2)
    ax3.set_xlabel('Epoch')
    ax3.set_ylabel('Loss')
    ax3.set_title('Adjacency Loss')
    ax3.legend()
    ax3.grid(True, alpha=0.3)

    # Contact component losses
    contact_components = list(train_losses['contact_components'].keys())
    for idx, component in enumerate(contact_components):
        row = (idx // 2) + 1
        col = (idx % 2) + 1
        if row == 1 and col == 2:  # Skip the cell we used for adjacency loss
            row = 2
            col = 0

        ax = fig.add_subplot(gs[row, col])
        ax.plot(train_losses['contact_components'][component], label='Train', linewidth=2)
        ax.plot(val_losses['contact_components'][component], label='Val', linewidth=2)
        ax.set_xlabel('Epoch')
        ax.set_ylabel('Loss')
        ax.set_title(f'{component.replace("_", " ").title()}')
        ax.legend()
        ax.grid(True, alpha=0.3)

    plt.tight_layout()
    return fig


# Generation
this is actually not a protein generation but a protein subsequence generation

In [ ]:
def generate_protein(model, adjacency_matrix, node_features, device, max_length=max_length, unique_aa=None):
    """
    Generate both a protein sequence and adjacency matrix using the dual output GRAN model

    Args:
        model: Trained DualOutputGRAN model
        adjacency_matrix: Input adjacency matrix
        node_features: Input node features
        device: Device to run inference on
        max_length: Maximum sequence length to generate
        unique_aa: List of amino acids for decoding

    Returns:
        Dictionary with generated sequence and adjacency matrix
    """
    # Prepare inputs
    if len(adjacency_matrix.shape) == 2:
        adjacency_matrix = adjacency_matrix.unsqueeze(0)
    if len(node_features.shape) == 2:
        node_features = node_features.unsqueeze(0)

    adjacency_matrix = adjacency_matrix.to(device)
    node_features = node_features.to(device)

    # Inference
    model.eval()
    with torch.no_grad():
        outputs = model(node_features, adjacency_matrix, max_length=max_length)

    # Decode sequence
    generated_ids = outputs['generated_sequence'][0]

    if unique_aa is None:
        amino_acids = "ACDEFGHIKLMNPQRSTVWYX"
    else:
        amino_acids = unique_aa

    protein_sequence = ""
    for aa_id in generated_ids:
        if aa_id.item() < len(amino_acids):
            protein_sequence += amino_acids[aa_id.item()]

    # Get predicted adjacency matrix
    predicted_adjacency = outputs['adjacency_matrix'][0].cpu().numpy()

    return {
        'protein_sequence': protein_sequence,
        'adjacency_matrix': predicted_adjacency
    }


In [ ]:
def generate_full_protein_from_subsequences(model, full_protein_length, subsequence_length,
                                            adjacency_matrix_template, node_features_template,
                                            device, UNIQUE_AA, overlap_strategy='average'):
    """
    Generate a full protein by combining multiple subsequences with proper three-letter code handling
    """
    print(f"\nGenerating full protein of length {full_protein_length}...")

    # Create an empty adjacency matrix for the full protein
    full_adjacency = np.zeros((full_protein_length, full_protein_length))
    full_sequence = [''] * full_protein_length

    # Track how many times each position has been filled (for averaging)
    if overlap_strategy == 'average':
        adjacency_counts = np.zeros((full_protein_length, full_protein_length))
        sequence_candidates = [[] for _ in range(full_protein_length)]

    # Step size for subsequences (controls overlap)
    step_size = subsequence_length // 2  # 50% overlap

    # List of valid amino acid three-letter codes
    valid_aa_codes = [
        'ALA', 'ARG', 'ASN', 'ASP', 'CYS', 'GLN', 'GLU', 'GLY', 'HIS', 'ILE',
        'LEU', 'LYS', 'MET', 'PHE', 'PRO', 'SER', 'THR', 'TRP', 'TYR', 'VAL', 'X'
    ]

    # Generate subsequences to cover the full protein
    for start_pos in range(0, full_protein_length, step_size):
        end_pos = min(start_pos + subsequence_length, full_protein_length)
        actual_subsequence_length = end_pos - start_pos

        # Prepare input for this subsequence
        subsequence_adj = adjacency_matrix_template[:actual_subsequence_length, :actual_subsequence_length]
        subsequence_features = node_features_template[:actual_subsequence_length]

        # Generate subsequence
        results = generate_protein(
            model, subsequence_adj, subsequence_features, device,
            max_length=actual_subsequence_length,
            unique_aa=UNIQUE_AA
        )

        # Parse the generated sequence into proper three-letter codes
        generated_raw_seq = results['protein_sequence']
        generated_adj = results['adjacency_matrix']

        # Use regular expressions to extract valid amino acid codes
        import re
        # Find all valid three-letter codes in the generated sequence
        generated_seq = []
        pos = 0
        while pos < len(generated_raw_seq):
            matched = False
            for code in valid_aa_codes:
                if generated_raw_seq[pos:].startswith(code):
                    generated_seq.append(code)
                    pos += len(code)
                    matched = True
                    break
            if not matched:
                # If no valid code found, skip a character
                pos += 1

        # Ensure we have the correct number of amino acids
        if len(generated_seq) < actual_subsequence_length:
            # Fill remaining positions with X
            generated_seq.extend(['X'] * (actual_subsequence_length - len(generated_seq)))
        elif len(generated_seq) > actual_subsequence_length:
            # Trim excess
            generated_seq = generated_seq[:actual_subsequence_length]

        # Handle sequence placement
        if overlap_strategy == 'average':
            for i, aa in enumerate(generated_seq):
                if start_pos + i < full_protein_length:
                    sequence_candidates[start_pos + i].append(aa)
        elif overlap_strategy == 'use_first':
            for i, aa in enumerate(generated_seq):
                if start_pos + i < full_protein_length and full_sequence[start_pos + i] == '':
                    full_sequence[start_pos + i] = aa
        elif overlap_strategy == 'use_last':
            for i, aa in enumerate(generated_seq):
                if start_pos + i < full_protein_length:
                    full_sequence[start_pos + i] = aa

        # Handle adjacency matrix placement
        for i in range(len(generated_seq)):
            for j in range(len(generated_seq)):
                if start_pos + i < full_protein_length and start_pos + j < full_protein_length:
                    if i < generated_adj.shape[0] and j < generated_adj.shape[1]:
                        if overlap_strategy == 'average':
                            full_adjacency[start_pos + i, start_pos + j] += generated_adj[i, j]
                            adjacency_counts[start_pos + i, start_pos + j] += 1
                        else:
                            full_adjacency[start_pos + i, start_pos + j] = generated_adj[i, j]

    # Post-process the results
    if overlap_strategy == 'average':
        # Average overlapping regions
        adjacency_counts[adjacency_counts == 0] = 1  # Avoid division by zero
        full_adjacency /= adjacency_counts

        # Choose most frequent amino acid for each position
        for i, candidates in enumerate(sequence_candidates):
            if candidates:
                # Count occurrences and pick most frequent
                from collections import Counter
                full_sequence[i] = Counter(candidates).most_common(1)[0][0]

    # Fill any remaining empty positions
    for i in range(full_protein_length):
        if full_sequence[i] == '':
            full_sequence[i] = 'X'

    # Join sequence with spaces for readability
    formatted_sequence = ' '.join(full_sequence)

    print(f"Generated sequence has {len(full_sequence)} amino acids")

    return {
        'full_sequence': formatted_sequence,
        'full_adjacency_matrix': full_adjacency
    }

In [ ]:

def generate_full_protein_sequence_and_structure(model, full_graphs, full_sequences, subgraphs,
                                                 subsequences, adjacency_matrices, node_features,
                                                 device, UNIQUE_AA):
    """Generate a full protein and visualize results"""
    if len(subgraphs) > 0:
        # Use the first subsequence to get chunk size
        reference_protein_length = len(subsequences[0])

        # Get the length of the first FULL protein, not subsequence
        if len(full_sequences) > 0:
            target_full_length = len(full_sequences[0])  # This is the actual full protein length
        else:
            # Fallback if full sequences not available
            target_full_length = 150  # Default value

        print(f"Target full protein length: {target_full_length}")
        print(f"Subsequence length: {reference_protein_length}")

        # Generate a full protein
        full_protein_results = generate_full_protein_from_subsequences(
            model=model,
            full_protein_length=target_full_length,
            subsequence_length=reference_protein_length,
            adjacency_matrix_template=adjacency_matrices[0],
            node_features_template=node_features[0],
            device=device,
            UNIQUE_AA=UNIQUE_AA,
            overlap_strategy='average'
        )

        print("\nGenerated full protein sequence:", full_protein_results['full_sequence'])
        print("Full adjacency matrix shape:", full_protein_results['full_adjacency_matrix'].shape)

        # Visualize and save results
        visualize_and_save_full_protein_structure(full_protein_results)

def visualize_and_save_full_protein_structure(full_protein_results):
    """Visualize the full protein structure and save to wandb"""
    # Convert adjacency to 3D coordinates using optimization
    configs = [{'lower': 0, 'upper': 8.0}]
    binary_adjacency = (full_protein_results['full_adjacency_matrix'] > 0.065).astype(np.float32)

    coords, fig = reconstruct_coords_local_plural_maps_colored(
        [binary_adjacency],  # List of contact maps
        configs,             # Distance constraints
        full_protein_results['full_sequence'],  # Amino acid sequence
        max_iter=2000,       # Optimization steps
        lr=0.01              # Learning rate
    )

    # Show visualization
    fig.show()

    # Save to wandb
    # 1. Save the interactive plotly figure

    # 2. Save the adjacency matrix as an image
    plt.figure(figsize=(8, 8))
    plt.imshow(full_protein_results['full_adjacency_matrix'], cmap='viridis')
    plt.title("Full Protein Adjacency Matrix")
    plt.colorbar()
    plt.close()

    # 3. Save the sequence as text

    # 4. Save coordinates as text file
    coords_file = "full_protein_coords.txt"
    np.savetxt(coords_file, coords, fmt='%.6f')

    # 5. Save visualization as HTML file
    html_file = "full_protein_structure.html"
    fig.write_html(html_file)

    print(f"All outputs saved to wandb")





In [ ]:


def reconstruct_coords_local_plural_maps_colored(contact_maps, configs, amino_acid_seq,
                                                 hard_mask=None, sharpness=10., window=25,
                                                 dim=3, lr=lr, max_iter=1000, prints=10):
    """
    Reconstruct 3D coordinates from multiple contact maps with different distance constraints.
    Visualize the structure with amino acid-based coloring.

    Args:
        contact_maps: List of binary contact maps
        configs: List of dictionaries containing 'lower' and 'upper' bounds for each map
        amino_acid_seq: List or string of amino acid sequence
        hard_mask: Boolean mask for valid positions
        sharpness: Parameter for contact prediction sharpness
        window: Window size for local interactions
        dim: Dimensionality of the coordinate space (default: 3D)
        lr: Learning rate for optimization
        max_iter: Maximum number of iterations
        prints: Number of progress updates

    Returns:
        Reconstructed coordinates as numpy array and Plotly figure
    """
    print('Gradient-based optimization with multiple contact maps...')

    # Define amino acid categories for coloring
    aa_categories = {
        # Hydrophobic
        'ALA': 'hydrophobic', 'VAL': 'hydrophobic', 'LEU': 'hydrophobic',
        'ILE': 'hydrophobic', 'PHE': 'hydrophobic', 'TRP': 'hydrophobic',
        'MET': 'hydrophobic', 'PRO': 'hydrophobic',
        # Polar
        'GLY': 'polar', 'SER': 'polar', 'THR': 'polar', 'CYS': 'polar',
        'TYR': 'polar', 'ASN': 'polar', 'GLN': 'polar',
        # Positively charged
        'LYS': 'positive', 'ARG': 'positive', 'HIS': 'positive',
        # Negatively charged
        'ASP': 'negative', 'GLU': 'negative',
        # Other
        'X': 'other', 'UNK': 'other'
    }

    # Color mapping
    color_dict = {
        'hydrophobic': 'blue',
        'polar': 'green',
        'positive': 'red',
        'negative': 'orange',
        'other': 'purple'
    }

    # Convert single letter to three letter if necessary
    if len(amino_acid_seq[0]) == 1:
        one_to_three = {
            'A': 'ALA', 'C': 'CYS', 'D': 'ASP', 'E': 'GLU',
            'F': 'PHE', 'G': 'GLY', 'H': 'HIS', 'I': 'ILE',
            'K': 'LYS', 'L': 'LEU', 'M': 'MET', 'N': 'ASN',
            'P': 'PRO', 'Q': 'GLN', 'R': 'ARG', 'S': 'SER',
            'T': 'THR', 'V': 'VAL', 'W': 'TRP', 'Y': 'TYR',
            'X': 'UNK'
        }
        amino_acid_seq = [one_to_three.get(aa, 'UNK') for aa in amino_acid_seq]

    # Create color list for amino acids
    aa_colors = [color_dict[aa_categories.get(aa, 'other')] for aa in amino_acid_seq]

    min_distance = 2.3  # Minimum allowed distance between points

    N = contact_maps[0].shape[0]
    # Ensure all contact maps have the same shape
    assert len(set([cmap.shape for cmap in contact_maps])) == 1, "Contact maps must have the same dimensions"

    # Initialize coordinates as a linear chain to provide better starting point
    coords = torch.zeros((N, dim))
    for i in range(N):
        coords[i, 0] = i * 3.8  # Typical C-alpha distance
        if dim > 1:
            coords[i, 1] = 2.0 * np.sin(i * 0.5)
        if dim > 2:
            coords[i, 2] = 2.0 * np.cos(i * 0.5)

    coords.requires_grad = True

    # Use a more robust optimizer and a learning rate scheduler
    optimizer = torch.optim.Adam([coords], lr=lr)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', factor=0.5, patience=patience)

    # Convert contact maps to tensors
    tensor_maps = [torch.tensor(contact_map, dtype=torch.float32) for contact_map in contact_maps]

    # Create repulsion mask (all non-diagonal elements)
    repulsion_mask = (torch.eye(N) == 0)

    if hard_mask is None:
        # Create a mask for local interactions
        local_mask = torch.zeros_like(tensor_maps[0])
        for i in range(N):
            for j in range(max(0, i - window), min(N, i + window + 1)):
                local_mask[i, j] = 1
    else:
        local_mask = torch.tensor(hard_mask.astype(bool))

    # Calculate positive weights for each contact map - ensure they're positive
    pos_weights = []
    for i, _ in enumerate(configs):
        # Make sure pos_weight is positive by using absolute value or a minimum bound
        ratio = max(1.0, local_mask.sum() / (tensor_maps[i].sum() + 1e-8))
        pos_weight = (ratio - 1.0) #.detach()
        pos_weight = max(1.0, pos_weight)  # Ensure positive weight
        pos_weights.append(pos_weight)

    # Best loss tracking for early stopping
    best_loss = float('inf')
    patience_counter = 0
    best_coords = coords.clone().detach()

    # Improved BCE loss function
    bce_loss_fn = torch.nn.BCEWithLogitsLoss(reduction='none')

    for step in range(max_iter):
        optimizer.zero_grad()

        # Calculate pairwise distances
        dists = torch.cdist(coords, coords, p=2)

        # Sequential connectivity loss - ensuring connected residues are close
        sequential_dists = torch.diag(dists, diagonal=1)
        connectivity_loss = torch.mean((sequential_dists - 3.8) ** 2)  # ~3.8Å is typical C-alpha distance

        # Calculate contact predictions for each config
        total_loss = torch.tensor(0.0, requires_grad=True)
        partial_losses = []

        for i, config in enumerate(configs):
            # Calculate logits for BCE loss
            if config['lower'] == 0:
                # For upper bound only, predict contact if distance < upper
                logits = sharpness * (config['upper'] - dists)
            else:
                # For range prediction, use a composite sigmoid
                upper_logits = sharpness * (config['upper'] - dists)
                lower_logits = sharpness * (dists - config['lower'])
                # Combine logits - both conditions must be true
                logits = torch.min(upper_logits, lower_logits)

            # Apply mask to focus on meaningful interactions
            weight_matrix = torch.ones_like(tensor_maps[i])
            weight_matrix = weight_matrix * pos_weights[i] * tensor_maps[i] + (1 - tensor_maps[i])

            # Calculate loss with weights
            map_loss = bce_loss_fn(logits, tensor_maps[i])
            weighted_loss = (map_loss * weight_matrix).mean()
            partial_losses.append(weighted_loss)
            total_loss = total_loss + weighted_loss

        # Add repulsion loss to maintain minimum distances
        min_dist_violation = torch.relu(min_distance - dists)
        repulsion_loss = (min_dist_violation[repulsion_mask] ** 2).sum()

        # Add all regularization terms
        total_loss = total_loss + 0.1 * repulsion_loss + 0.05 * connectivity_loss

        # Backwards pass and optimization
        total_loss.backward()

        # Gradient clipping to prevent instability
        torch.nn.utils.clip_grad_norm_([coords], max_norm=1.0)

        optimizer.step()
        scheduler.step(total_loss)

        # Print progress and update best model if needed
        if step % max(1, int(max_iter // prints)) == 0:
            partial_losses_str = [f"{p_loss.item():8.2f}" for p_loss in partial_losses]
            print(f"Step {step:7d} | total loss: {total_loss.item():8.2f} ({', '.join(partial_losses_str)})")

            if total_loss.item() < best_loss:
                best_loss = total_loss.item()
                best_coords = coords.clone().detach()
                patience_counter = 0
            else:
                patience_counter += 1

            # Early stopping
            if patience_counter > 100:
                print("Early stopping triggered.")
                break

    # Final loss report
    partial_losses_str = [f"{p_loss.item():8.2f}" for p_loss in partial_losses]
    print(f"Step {step:7d} | total loss: {total_loss.item():8.2f} ({', '.join(partial_losses_str)}), final.")
    print('')

    # Get the coordinates
    final_coords = best_coords.detach().numpy()

    # Create visualization
    fig = visualize_protein_structure(final_coords, amino_acid_seq, aa_colors)

    return final_coords, fig

def visualize_protein_structure(coords, amino_acid_seq, colors=None):
    """
    Visualize protein structure using Plotly

    Args:
        coords: 3D coordinates of the protein structure (N x 3)
        amino_acid_seq: List of amino acid types
        colors: List of colors for each amino acid

    Returns:
        Plotly figure object
    """
    N = coords.shape[0]

    # Use rainbow colors if no colors provided
    if colors is None:
        colors = cm.rainbow(np.linspace(0, 1, N))
        colors = [f'rgb({int(r*255)},{int(g*255)},{int(b*255)})' for r, g, b, _ in colors]

    # Ensure colors is a list of strings
    if isinstance(colors[0], str):
        color_strings = colors
    else:
        color_strings = [f'rgb({int(r*255)},{int(g*255)},{int(b*255)})' for r, g, b, _ in colors]

    # Create figure
    fig = go.Figure()

    # Add markers for amino acids
    fig.add_trace(go.Scatter3d(
        x=coords[:, 0],
        y=coords[:, 1],
        z=coords[:, 2],
        mode='markers',
        marker=dict(
            size=6,
            color=color_strings,
            opacity=0.8
        ),
        text=[f"{i+1}: {aa}" for i, aa in enumerate(amino_acid_seq)],
        hoverinfo='text',
        name='Amino acids'
    ))

    # Add lines connecting sequential residues
    x_lines, y_lines, z_lines = [], [], []
    line_colors = []

    for i in range(N-1):
        # Add line segments connecting adjacent residues
        x_lines.extend([coords[i, 0], coords[i+1, 0], None])
        y_lines.extend([coords[i, 1], coords[i+1, 1], None])
        z_lines.extend([coords[i, 2], coords[i+1, 2], None])
        line_colors.extend([color_strings[i], color_strings[i+1], 'rgba(0,0,0,0)'])

    fig.add_trace(go.Scatter3d(
        x=x_lines,
        y=y_lines,
        z=z_lines,
        mode='lines',
        line=dict(
            color=line_colors,
            width=3
        ),
        hoverinfo='none',
        name='Backbone'
    ))

    # Configure layout
    fig.update_layout(
        title="Protein Structure",
        scene=dict(
            xaxis=dict(visible=False, showticklabels=False),
            yaxis=dict(visible=False, showticklabels=False),
            zaxis=dict(visible=False, showticklabels=False),
            bgcolor='rgba(255,255,255,1)'
        ),
        showlegend=False,
        margin=dict(l=0, r=0, b=0, t=30),
        paper_bgcolor='rgba(0,0,0,0)',
        scene_camera=dict(
            eye=dict(x=1.2, y=1.2, z=1.2)
        )
    )

    return fig



# Main Pipeline Run of model

In [ ]:
def get_dataloaders(aa_sequences, adjacency_matrices, node_features, batch_size=32):
    """Create train and validation dataloaders"""
    split_idx = int(0.8 * len(aa_sequences))

    train_aa = aa_sequences[:split_idx]
    train_adj = adjacency_matrices[:split_idx]
    train_nf = node_features[:split_idx]

    val_aa = aa_sequences[split_idx:]
    val_adj = adjacency_matrices[split_idx:]
    val_nf = node_features[split_idx:]

    train_loader = create_dataloader(train_aa, train_adj, train_nf, batch_size=batch_size)
    val_loader = create_dataloader(val_aa, val_adj, val_nf, batch_size=batch_size)

    return train_loader, val_loader, train_aa, train_adj, train_nf

def print_and_get_model(node_features, UNIQUE_AA, device, model_path):
    """Print model configuration and create the model"""
    node_features_dim = node_features[0].size(1)

    # The feature dimension is now 38: Meiler(7) + AA one-hot(22) + SS one-hot(9)
    # But we keep the same architecture for processing
    hidden_dim = 128  # Fixed dimension to ensure compatibility
    num_layers = 2
    n_heads = 4
    amino_acid_vocab_size = len(UNIQUE_AA)

    print(f"Model configuration:")
    print(f"- Node features dimension: {node_features_dim}")
    print(f"- Hidden dimension: {hidden_dim}")
    print(f"- Number of graph layers: {num_layers}")
    print(f"- Number of attention heads: {n_heads}")
    print(f"- Amino acid vocabulary size: {amino_acid_vocab_size}")

    # Create the dual output model
    model = DualOutputGRAN(
        node_features=node_features_dim,  # Now 38
        hidden_dim=hidden_dim,
        num_layers=num_layers,
        n_heads=n_heads,
        dropout=0.1,
        amino_acid_vocab_size=amino_acid_vocab_size
    ).to(device)

    # Load model if it exists (for continuing training)
    if os.path.exists(model_path):
        print(f"Found existing model checkpoint at {model_path}. Loading...")
        model = load_model(model_path, model, device)
    else:
        print("No existing model found. Starting training from scratch.")

    return model
def sanity_check_forward_pass(model, train_nf, train_adj, train_aa, device):
    """Test forward pass with sample data"""
    print("Testing model with sample data...")
    with torch.no_grad():
        # Create small test batch
        test_node_feats = train_nf[0].unsqueeze(0).to(device)
        test_adj = train_adj[0].unsqueeze(0).to(device)
        test_seq = train_aa[0].unsqueeze(0).to(device)

        # Forward pass
        outputs = model(test_node_feats, test_adj, test_seq)
        print("Forward pass successful!")
        print(f"Sequence logits shape: {outputs['sequence_logits'].shape}")
        print(f"Adjacency matrix shape: {outputs['adjacency_matrix'].shape}")

def train_model(model, train_loader, val_loader, device, num_epochs, lr, patience, min_epochs, model_path):
    """Train the model and return loss histories"""
    print("\nStarting model training...")
    train_losses, val_losses = train_dual_output_model_enhanced(
        model, train_loader, val_loader,
        num_epochs=num_epochs,
        lr=lr,
        device=device,
        patience=patience,
        min_epochs=min_epochs,
        checkpoint_path=model_path
    )
    return train_losses, val_losses

def plot_train_results(train_losses, val_losses):
    """Plot training results and save to wandb"""
    plt.figure(figsize=(15, 5))

    # Sequence loss plot
    plt.subplot(1, 3, 1)
    plt.plot(train_losses['sequence'], label='Train Loss')
    plt.plot(val_losses['sequence'], label='Val Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Sequence Loss')
    plt.title('Sequence Generation')
    plt.legend()

    # Adjacency loss plot
    plt.subplot(1, 3, 2)
    plt.plot(train_losses['adjacency'], label='Train Loss')
    plt.plot(val_losses['adjacency'], label='Val Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Adjacency Loss')
    plt.title('Adjacency Prediction')
    plt.legend()

    # Combined loss plot
    plt.subplot(1, 3, 3)
    plt.plot(train_losses['combined'], label='Train Loss')
    plt.plot(val_losses['combined'], label='Val Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Combined Loss')
    plt.title('Combined Performance')
    plt.legend()

    plt.tight_layout()

    # Save to wandb

    # Also save individual loss series as plots
    for loss_type in ['sequence', 'adjacency', 'combined']:
        plt.figure(figsize=(8, 6))
        plt.plot(train_losses[loss_type], label='Train Loss', linewidth=2)
        plt.plot(val_losses[loss_type], label='Val Loss', linewidth=2)
        plt.xlabel('Epoch')
        plt.ylabel(f'{loss_type.capitalize()} Loss')
        plt.title(f'{loss_type.capitalize()} Loss Progression')
        plt.legend()
        plt.grid(True, alpha=0.3)
        plt.close()

    # Save loss data as CSV for further analysis
    import pandas as pd
    loss_df = pd.DataFrame({
        'epoch': range(len(train_losses['combined'])),
        'train_sequence': train_losses['sequence'],
        'val_sequence': val_losses['sequence'],
        'train_adjacency': train_losses['adjacency'],
        'val_adjacency': val_losses['adjacency'],
        'train_combined': train_losses['combined'],
        'val_combined': val_losses['combined']
    })

    # Save CSV to wandb
    csv_filename = "loss_history.csv"
    loss_df.to_csv(csv_filename, index=False)

    # Log the final losses as summary metrics

    plt.show()

def generate_sample_protein_sequence_and_structure(model, subgraphs, subsequences, adjacency_matrices,
                                                   node_features, device, UNIQUE_AA):
    """Generate a sample protein and visualize results"""
    if len(subgraphs) > 0:
        print("\nGenerating sample protein...")
        sample_seq = subsequences[0]

        # Convert to tensors for generation
        sample_adj = adjacency_matrices[0].detach().clone()
        sample_features = node_features[0].detach().clone()

        # Generate protein with dual outputs
        results = generate_protein(
            model, sample_adj, sample_features, device,
            max_length=len(sample_seq),
            unique_aa=UNIQUE_AA
        )

        print("\nOriginal sequence:", ''.join(sample_seq[:20]) + "..." if len(sample_seq) > 20 else ''.join(sample_seq))
        print("Generated sequence:", results['protein_sequence'])

        print("Generated adjacency matrix shape:", results['adjacency_matrix'].shape)

        # Visualize original vs generated adjacency matrices
        plt.figure(figsize=(12, 5))

        plt.subplot(1, 2, 1)
        plt.imshow(adjacency_matrices[0].numpy(), cmap='viridis')
        plt.title("Original Adjacency Matrix")
        plt.colorbar()

        plt.subplot(1, 2, 2)
        plt.imshow(results['adjacency_matrix'], cmap='viridis')
        plt.title("Generated Adjacency Matrix")
        plt.colorbar()

        plt.tight_layout()

        # Save visualization to wandb
        plt.show()

        # Convert adjacency matrices to binary for comparison
        orig_adj_binary = (adjacency_matrices[0].numpy() > 0.5).astype(float)
        pred_adj_binary = (results['adjacency_matrix'] > 0.5).astype(float)

        # Mask diagonal elements
        mask = np.ones_like(orig_adj_binary) - np.eye(orig_adj_binary.shape[0])
        adj_match_percent = np.sum((orig_adj_binary == pred_adj_binary) * mask) / np.sum(mask) * 100

        print(f"Adjacency matrix match accuracy: {adj_match_percent:.2f}%")

        # Create detailed adjacency matrices comparison plots
        plt.figure(figsize=(15, 5))

        # Original adjacency (binary)
        plt.subplot(1, 3, 1)
        plt.imshow(orig_adj_binary, cmap='gray')
        plt.title("Original Adjacency (Binary)")
        plt.colorbar()

        # Generated adjacency (binary)
        plt.subplot(1, 3, 2)
        plt.imshow(pred_adj_binary, cmap='gray')
        plt.title("Generated Adjacency (Binary)")
        plt.colorbar()

        # Difference map
        plt.subplot(1, 3, 3)
        diff = orig_adj_binary - pred_adj_binary
        plt.imshow(diff, cmap='RdBu', vmin=-1, vmax=1)
        plt.title("Difference (Blue=FalsePositive, Red=FalseNegative)")
        plt.colorbar()

        plt.tight_layout()
        plt.show()

    # Save model
    torch.save(model.state_dict(), "dual_output_gran_model.pt")
    print("\nModel saved to dual_output_gran_model.pt")


# Modified main function that calls the individual methods
def main():
    # Get dataloaders
    train_loader, val_loader, train_aa, train_adj, train_nf = get_dataloaders(
        aa_sequences, adjacency_matrices, node_features, batch_size=32
    )

    # Create and configure model
    model = print_and_get_model(node_features, UNIQUE_AA, device, model_path)

    # Test forward pass
    sanity_check_forward_pass(model, train_nf, train_adj, train_aa, device)

    # Train model
    train_losses, val_losses = train_model(
        model, train_loader, val_loader, device,
        num_epochs=num_epochs, lr=lr, patience=patience,
        min_epochs=min_epochs, model_path=model_path
    )

    # Plot results
    plot_train_results_enhanced(train_losses, val_losses)

    # Generate sample protein
    generate_sample_protein_sequence_and_structure(
        model, subgraphs, subsequences, adjacency_matrices,
        node_features, device, UNIQUE_AA
    )

    # Generate full protein (not just subsequence)
    generate_full_protein_sequence_and_structure(
        model, full_graphs, full_sequences, subgraphs, subsequences,
        adjacency_matrices, node_features, device, UNIQUE_AA
    )


if __name__ == "__main__":
    main()

In [ ]:
# Step 1: Load the trained model and generate protein sequence and adjacency matrix
def generate_from_model(model_path, sample_adjacency, sample_features, device, unique_aa, max_length=max_length):
    """
    Generate a protein sequence and adjacency matrix from a trained model

    Args:
        model_path: Path to the saved model weights
        sample_adjacency: Sample adjacency matrix to use as input
        sample_features: Sample node features to use as input
        device: Device to run inference on (cpu or gpu)
        unique_aa: List of unique amino acids for decoding

    Returns:
        Dictionary with generated sequence and adjacency matrix
    """
    # Create model with the same architecture as during training
    node_features_dim = sample_features.size(1)
    model = DualOutputGRAN(
        node_features=node_features_dim,
        hidden_dim=128,
        num_layers=2,
        n_heads=4,
        dropout=0.1,
        amino_acid_vocab_size=len(unique_aa)
    ).to(device)

    # Load the saved weights
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.eval()

    # Generate using the same function we defined earlier
    results = generate_protein(
        model, sample_adjacency, sample_features, device,
        max_length=max_length,  # Adjust as needed
        unique_aa=unique_aa
    )

    print("Generated protein sequence:", results['protein_sequence'])
    print("Generated adjacency matrix shape:", results['adjacency_matrix'].shape)

    return results

# Step 2: Convert the generated outputs to 3D structure and visualize
def structure_from_generated_output(generated_results):
    """
    Convert generated model output to 3D protein structure

    Args:
        generated_results: Output from generate_from_model

    Returns:
        3D coordinates and visualization figure
    """
    # Extract results
    sequence = generated_results['protein_sequence']
    adjacency = generated_results['adjacency_matrix']

    # Convert adjacency to binary contact map with threshold
    binary_adjacency = (adjacency > 0.065).astype(np.float32)

    # Define distance constraints
    configs = [{'lower': 0, 'upper': 8.0}]

    # Generate 3D coordinates and visualization
    coords, fig = reconstruct_coords_local_plural_maps_colored(
        [binary_adjacency],  # List of contact maps
        configs,             # Distance constraints
        sequence,            # Amino acid sequence
        max_iter=2000,       # Optimization steps
        lr=0.01              # Learning rate
    )

    return coords, fig

def main_visualization_pipeline():

    #sample_adj = torch.tensor(adjacency_matrices[0], dtype=torch.float32)
    #sample_features = torch.tensor(node_features[0], dtype=torch.float32)
    sample_adj = adjacency_matrices[0].detach().clone()
    sample_features = node_features[0].detach().clone()

    # Step 1: Generate from model
    generated_results = generate_from_model(
        model_path, sample_adj, sample_features, device, UNIQUE_AA
    )

    # Step 2: Convert to 3D structure
    coords, fig = structure_from_generated_output(generated_results)

    # Show visualization
    fig.show()

    # Optional: Save visualization
    fig.write_html("protein_structure.html")

    return coords, fig, generated_results

# Run the pipeline
coords, fig, generated_results = main_visualization_pipeline()

In [ ]:
def visualize_original_vs_generated(original_seq, original_adj, generated_seq, generated_adj):
    """
    Create a side-by-side visualization of original and generated protein structures
    """
    # Convert adjacency matrices to binary contact maps if needed
    binary_orig_adj = (original_adj > 0.5).astype(np.float32)
    binary_gen_adj = (generated_adj > 0.065).astype(np.float32)  # Adjust threshold as needed

    # Define distance constraints
    configs = [{'lower': 0, 'upper': 8.0}]

    # Generate 3D coordinates for both structures
    orig_coords, _ = reconstruct_coords_local_plural_maps_colored(
        [binary_orig_adj], configs, original_seq, max_iter=2000, lr=0.01
    )

    gen_coords, _ = reconstruct_coords_local_plural_maps_colored(
        [binary_gen_adj], configs, generated_seq, max_iter=2000, lr=0.01
    )

    # Create combined visualization
    fig = go.Figure()

    # Generate color mappings for amino acids
    def get_aa_colors(sequence):
        aa_categories = {
            # Simplified categories for visual distinction
            'A': 'hydrophobic', 'V': 'hydrophobic', 'L': 'hydrophobic',
            'I': 'hydrophobic', 'F': 'hydrophobic', 'W': 'hydrophobic',
            'M': 'hydrophobic', 'P': 'hydrophobic',
            'G': 'polar', 'S': 'polar', 'T': 'polar', 'C': 'polar',
            'Y': 'polar', 'N': 'polar', 'Q': 'polar',
            'K': 'positive', 'R': 'positive', 'H': 'positive',
            'D': 'negative', 'E': 'negative',
        }

        color_dict = {
            'hydrophobic': 'blue',
            'polar': 'green',
            'positive': 'red',
            'negative': 'orange',
            'other': 'purple'
        }

        return [color_dict.get(aa_categories.get(aa, 'other'), 'purple') for aa in sequence]

    orig_colors = get_aa_colors(original_seq)
    gen_colors = get_aa_colors(generated_seq)

    # Add original structure (left side)
    fig.add_trace(go.Scatter3d(
        x=orig_coords[:, 0],
        y=orig_coords[:, 1],
        z=orig_coords[:, 2],
        mode='markers+lines',
        marker=dict(
            size=5,
            color=orig_colors,
            opacity=0.8
        ),
        line=dict(
            color='lightblue',
            width=2
        ),
        text=[f"Original {i+1}: {aa}" for i, aa in enumerate(original_seq)],
        hoverinfo='text',
        name='Original',
        scene='scene1'
    ))

    # Add generated structure (right side)
    fig.add_trace(go.Scatter3d(
        x=gen_coords[:, 0],
        y=gen_coords[:, 1],
        z=gen_coords[:, 2],
        mode='markers+lines',
        marker=dict(
            size=5,
            color=gen_colors,
            opacity=0.8
        ),
        line=dict(
            color='lightgreen',
            width=2
        ),
        text=[f"Generated {i+1}: {aa}" for i, aa in enumerate(generated_seq)],
        hoverinfo='text',
        name='Generated',
        scene='scene2'
    ))

    # Configure layout with two 3D scenes
    fig.update_layout(
        title="Original vs Generated Protein Structure",
        scene1=dict(
            domain=dict(x=[0, 0.5], y=[0, 1]),
            xaxis=dict(visible=False),
            yaxis=dict(visible=False),
            zaxis=dict(visible=False),
            aspectmode='cube',
            camera=dict(eye=dict(x=1.2, y=1.2, z=1.2))
        ),
        scene2=dict(
            domain=dict(x=[0.5, 1], y=[0, 1]),
            xaxis=dict(visible=False),
            yaxis=dict(visible=False),
            zaxis=dict(visible=False),
            aspectmode='cube',
            camera=dict(eye=dict(x=1.2, y=1.2, z=1.2))
        ),
        margin=dict(l=0, r=0, b=0, t=40),
        paper_bgcolor='white',
        showlegend=False,
    )

    return fig



In [ ]:


# Sample input from validation set (or any other source)

sample_adj = adjacency_matrices[0].detach().clone()
sample_features = node_features[0].detach().clone()
# Step 1: Generate from model
generated_results = generate_from_model(
    model_path, sample_adj, sample_features, device, UNIQUE_AA, max_length=max_length
)


In [ ]:
def generate_protein_sample(adjacency_matrices, node_features, subsequences, model_path, unique_aa, device, sample_index=None, max_length=max_length):
    """
    Generate a protein sample using the trained model and return both original and generated data

    Args:
        adjacency_matrices: List of adjacency matrices
        node_features: List of node features
        subsequences: List of amino acid sequences
        model_path: Path to the saved model weights
        unique_aa: List of unique amino acids
        device: Device to run on (cpu/gpu)
        sample_index: Optional specific index to use, otherwise random

    Returns:
        Dictionary containing:
            - original_seq: Original amino acid sequence
            - original_adj: Original adjacency matrix
            - generated_seq: Generated amino acid sequence
            - generated_adj: Generated adjacency matrix
    """
    # Select a random sample if index not provided
    if sample_index is None:
        sample_index = random.randint(0, len(adjacency_matrices)-1)

    print(f"Using sample index: {sample_index}")

    # Prepare sample input
    sample_adj = adjacency_matrices[sample_index].detach().clone()
    sample_features = node_features[sample_index].detach().clone()

    # Generate from model
    generated_results = generate_from_model(
        model_path, sample_adj, sample_features, device, unique_aa, max_length=max_length
    )

    # Get original data
    original_seq = subsequences[sample_index]
    original_adj = adjacency_matrices[sample_index].numpy()

    # Get generated data
    generated_seq = generated_results['protein_sequence']
    generated_adj = generated_results['adjacency_matrix']

    # Calculate similarity
    def three_to_one_letter(three_letter):
        conversion = {
            'ALA': 'A', 'ARG': 'R', 'ASN': 'N', 'ASP': 'D', 'CYS': 'C',
            'GLN': 'Q', 'GLU': 'E', 'GLY': 'G', 'HIS': 'H', 'ILE': 'I',
            'LEU': 'L', 'LYS': 'K', 'MET': 'M', 'PHE': 'F', 'PRO': 'P',
            'SER': 'S', 'THR': 'T', 'TRP': 'W', 'TYR': 'Y', 'VAL': 'V',
            'X': 'X', 'UNK': 'X'
        }
        return conversion.get(three_letter, 'X')

    # Convert original sequence to one-letter codes if needed
    if len(original_seq) > 0 and len(original_seq[0]) == 3:  # 3-letter code
        original_seq_1letter = [three_to_one_letter(aa) for aa in original_seq]
    else:  # Already 1-letter code
        original_seq_1letter = original_seq

    # Calculate sequence similarity
    common_length = min(len(original_seq_1letter), len(generated_seq))
    if common_length > 0:
        seq_match_count = sum([1 if a == b else 0 for a, b in
                               zip(original_seq_1letter[:common_length],
                                   generated_seq[:common_length])])
        seq_match_percent = (seq_match_count / common_length) * 100
    else:
        seq_match_percent = 0.0

    print(f"Sequence similarity: {seq_match_percent:.2f}%")

    return {
        'original_seq': original_seq,
        'original_adj': original_adj,
        'generated_seq': generated_seq,
        'generated_adj': generated_adj,
        'similarity': seq_match_percent,
        'sample_index': sample_index
    }

In [ ]:
# Example usage
result = generate_protein_sample(
    adjacency_matrices,
    node_features,
    subsequences,
    model_path,
    UNIQUE_AA,
    device, max_length=max_length
)

# Visualize the results
fig = visualize_original_vs_generated(
    original_seq=result['original_seq'],
    original_adj=result['original_adj'],
    generated_seq=result['generated_seq'],
    generated_adj=result['generated_adj']
)
fig.show()



In [ ]:
def visualize_original_vs_generated2(original_seq, original_adj, generated_seq, generated_adj):
    """
    Create a side-by-side visualization of original and generated protein structures
    with improved amino acid coloring
    """
    # Convert adjacency matrices to binary contact maps if needed
    binary_orig_adj = (original_adj > 0.5).astype(np.float32)
    binary_gen_adj = (generated_adj > 0.065).astype(np.float32)  # Adjust threshold as needed

    # Define distance constraints
    configs = [{'lower': 0, 'upper': 8.0}]

    # Detailed amino acid color scheme based on physicochemical properties
    aa_colors = {
        # Hydrophobic (blue shades)
        'A': '#0000FF',  # Blue
        'V': '#000080',  # Navy
        'L': '#4169E1',  # Royal Blue
        'I': '#1E90FF',  # Dodger Blue
        'M': '#00BFFF',  # Deep Sky Blue
        'F': '#87CEEB',  # Sky Blue
        'W': '#B0C4DE',  # Light Steel Blue

        # Polar (green shades)
        'S': '#008000',  # Green
        'T': '#006400',  # Dark Green
        'N': '#32CD32',  # Lime Green
        'Q': '#00FF00',  # Lime
        'Y': '#98FB98',  # Pale Green
        'C': '#90EE90',  # Light Green
        'G': '#ADFF2F',  # Green Yellow

        # Positively charged (red shades)
        'K': '#FF0000',  # Red
        'R': '#B22222',  # Fire Brick
        'H': '#FF6347',  # Tomato

        # Negatively charged (orange/yellow shades)
        'D': '#FFA500',  # Orange
        'E': '#FFD700',  # Gold

        # Special
        'P': '#800080',  # Purple
        'X': '#A9A9A9',  # Dark Gray
    }

    # Convert 3-letter AA codes to 1-letter if needed
    three_to_one = {
        'ALA': 'A', 'ARG': 'R', 'ASN': 'N', 'ASP': 'D', 'CYS': 'C',
        'GLN': 'Q', 'GLU': 'E', 'GLY': 'G', 'HIS': 'H', 'ILE': 'I',
        'LEU': 'L', 'LYS': 'K', 'MET': 'M', 'PHE': 'F', 'PRO': 'P',
        'SER': 'S', 'THR': 'T', 'TRP': 'W', 'TYR': 'Y', 'VAL': 'V',
        'X': 'X', 'UNK': 'X'
    }

    # Process original sequence
    if len(original_seq) > 0 and len(original_seq[0]) == 3:  # 3-letter code
        orig_seq_1letter = [three_to_one.get(aa, 'X') for aa in original_seq]
    else:  # Already 1-letter code
        orig_seq_1letter = original_seq

    # Generate 3D coordinates for both structures
    print("Generating 3D structure for original protein...")
    orig_coords_results, orig_fig = reconstruct_coords_local_plural_maps_colored(
        [binary_orig_adj], configs, original_seq, hard_mask=None, sharpness=15.,
        window=25, dim=3, lr=1e-2, max_iter=2000, prints=5
    )

    print("Generating 3D structure for model-generated protein...")
    gen_coords_results, gen_fig = reconstruct_coords_local_plural_maps_colored(
        [binary_gen_adj], configs, generated_seq, hard_mask=None, sharpness=15.,
        window=25, dim=3, lr=1e-2, max_iter=2000, prints=5
    )

    # Use the coordinates from the results
    orig_coords = orig_coords_results
    gen_coords = gen_coords_results

    # Create combined visualization
    fig = go.Figure()

    # Add original structure (left side)
    # Nodes (amino acids)
    orig_colors = [aa_colors.get(aa, '#808080') for aa in orig_seq_1letter]

    fig.add_trace(go.Scatter3d(
        x=orig_coords[:, 0],
        y=orig_coords[:, 1],
        z=orig_coords[:, 2],
        mode='markers',
        marker=dict(
            size=6,
            color=orig_colors,
            opacity=0.8
        ),
        text=[f"Orig {i+1}: {aa}" for i, aa in enumerate(orig_seq_1letter)],
        hoverinfo='text',
        name='Original',
        scene='scene1'
    ))

    # Lines connecting sequential residues
    x_lines, y_lines, z_lines = [], [], []
    line_colors = []

    for i in range(len(orig_coords)-1):
        # Add line segments
        x_lines.extend([orig_coords[i, 0], orig_coords[i+1, 0], None])
        y_lines.extend([orig_coords[i, 1], orig_coords[i+1, 1], None])
        z_lines.extend([orig_coords[i, 2], orig_coords[i+1, 2], None])
        line_colors.extend([orig_colors[i], orig_colors[i+1], 'rgba(0,0,0,0)'])

    fig.add_trace(go.Scatter3d(
        x=x_lines,
        y=y_lines,
        z=z_lines,
        mode='lines',
        line=dict(
            color='silver',
            width=2
        ),
        hoverinfo='none',
        showlegend=False,
        scene='scene1'
    ))

    # Add generated structure (right side)
    # Process generated sequence
    gen_colors = [aa_colors.get(aa, '#808080') for aa in generated_seq]

    fig.add_trace(go.Scatter3d(
        x=gen_coords[:, 0],
        y=gen_coords[:, 1],
        z=gen_coords[:, 2],
        mode='markers',
        marker=dict(
            size=6,
            color=gen_colors,
            opacity=0.8
        ),
        text=[f"Gen {i+1}: {aa}" for i, aa in enumerate(generated_seq)],
        hoverinfo='text',
        name='Generated',
        scene='scene2'
    ))

    # Lines for generated structure
    x_lines, y_lines, z_lines = [], [], []

    for i in range(len(gen_coords)-1):
        # Add line segments
        x_lines.extend([gen_coords[i, 0], gen_coords[i+1, 0], None])
        y_lines.extend([gen_coords[i, 1], gen_coords[i+1, 1], None])
        z_lines.extend([gen_coords[i, 2], gen_coords[i+1, 2], None])

    fig.add_trace(go.Scatter3d(
        x=x_lines,
        y=y_lines,
        z=z_lines,
        mode='lines',
        line=dict(
            color='silver',
            width=2
        ),
        hoverinfo='none',
        showlegend=False,
        scene='scene2'
    ))

    # Add a legend for amino acid types
    amino_groups = {
        'Hydrophobic': ['A', 'V', 'L', 'I', 'M', 'F', 'W'],
        'Polar': ['S', 'T', 'N', 'Q', 'Y', 'C', 'G'],
        'Positively charged': ['K', 'R', 'H'],
        'Negatively charged': ['D', 'E'],
        'Special': ['P'],
        'Unknown': ['X']
    }

    # Add legend traces (invisible points with the right colors)
    for group_name, aas in amino_groups.items():
        for aa in aas:
            fig.add_trace(go.Scatter3d(
                x=[None], y=[None], z=[None],
                mode='markers',
                marker=dict(
                    size=6,
                    color=aa_colors.get(aa, '#808080'),
                    opacity=0.8
                ),
                name=f"{group_name}: {aa}",
                scene='scene1'  # Associate with first scene
            ))

    # Configure layout with two 3D scenes
    fig.update_layout(
        title="Original vs Generated Protein Structure",
        scene1=dict(
            domain=dict(x=[0, 0.5], y=[0, 1]),
            xaxis=dict(visible=False),
            yaxis=dict(visible=False),
            zaxis=dict(visible=False),
            aspectmode='cube',
            camera=dict(eye=dict(x=1.2, y=1.2, z=1.2))
        ),
        scene2=dict(
            domain=dict(x=[0.5, 1], y=[0, 1]),
            xaxis=dict(visible=False),
            yaxis=dict(visible=False),
            zaxis=dict(visible=False),
            aspectmode='cube',
            camera=dict(eye=dict(x=1.2, y=1.2, z=1.2))
        ),
        margin=dict(l=0, r=0, b=0, t=40),
        paper_bgcolor='white',
        legend=dict(
            x=0.01,
            y=0.99,
            traceorder='normal',
            bgcolor='rgba(255,255,255,0.5)',
            bordercolor='rgba(0,0,0,0.5)',
            borderwidth=1
        )
    )

    # Add sequence information as annotations
    fig.add_annotation(
        x=0.25, y=1.05,
        xref='paper', yref='paper',
        text=f"Original: {original_seq[:15]}..." if len(original_seq) > 15 else original_seq,
        showarrow=False,
        font=dict(size=10)
    )

    fig.add_annotation(
        x=0.75, y=1.05,
        xref='paper', yref='paper',
        text=f"Generated: {generated_seq[:15]}..." if len(generated_seq) > 15 else generated_seq,
        showarrow=False,
        font=dict(size=10)
    )

    return fig

In [ ]:
# Example usage
result = generate_protein_sample(
    adjacency_matrices,
    node_features,
    subsequences,
    model_path,
    UNIQUE_AA,
    device, max_length=max_length
)

# Visualize the results
fig = visualize_original_vs_generated2(
    original_seq=result['original_seq'],
    original_adj=result['original_adj'],
    generated_seq=result['generated_seq'],
    generated_adj=result['generated_adj']
)
fig.show()
